#INGEST FACT RETURNS & SHIPMENTS RAW DATA

##FACT_ORDER_RETURNS

In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "storageaccount", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
adls_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net/"
    f"order_returns/landing/"
)

bronze_checkpoint_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net/"
    f"checkpoint/bronze/fact_order_returns/"
)

In [0]:
(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", bronze_checkpoint_path)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("pathGlobFilter", "*.csv")
    .load(adls_path)

    .withColumn("ingest_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))

    .writeStream
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(f"{catalog_name}.bronze.brz_order_returns")
    .awaitTermination()
)

In [0]:
# display(
#     spark.sql(
#         f"""
#         SELECT COUNT(*)
#         FROM {catalog_name}.bronze.brz_order_returns
#         """
#     )
# )

# display(
#     spark.sql(
#         f"""
#         SELECT MIN(return_ts),
#                MAX(return_ts)
#         FROM {catalog_name}.bronze.brz_order_returns
#         """
#     )
# )

COUNT(*)
20194


MIN(return_ts),MAX(return_ts)
2024-01-08T00:52:46.000Z,2025-08-31T22:47:47.000Z


In [0]:
# df = spark.table(f"{catalog_name}.bronze.brz_order_returns")

# df.printSchema()
# display(df.limit(20))

root
 |-- return_ts: timestamp (nullable = true)
 |-- order_dt: date (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- reason: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



return_ts,order_dt,order_id,reason,_rescued_data,ingest_timestamp,source_file
2024-05-01T00:02:45.000Z,2024-04-20,122138,Wrong Item,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T01:27:25.000Z,2024-04-21,123994,Wrong Item,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T02:24:11.000Z,2024-04-19,120440,Other,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T02:45:13.000Z,2024-04-21,123548,Quality Issue,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T03:12:47.000Z,2024-04-21,123925,Wrong Item,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T03:32:46.000Z,2024-04-19,120365,Other,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T04:24:57.000Z,2024-04-16,117331,Late Delivery,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T05:33:52.000Z,2024-04-21,123702,Late Delivery,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T07:06:09.000Z,2024-04-18,119335,Late Delivery,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T07:15:25.000Z,2024-04-20,122387,Damaged,null,2026-06-15T07:02:20.249Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv


##FACT_ORDER_SHIPMENTS

In [0]:
adls_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net/"
    f"order_shipments/landing/"
)

In [0]:
bronze_checkpoint_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net/"
    f"checkpoint/bronze/fact_order_shipments/"
)

In [0]:
(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", bronze_checkpoint_path)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("pathGlobFilter", "*.csv")
    .load(adls_path)

    .withColumn("ingest_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))

    .writeStream
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(f"{catalog_name}.bronze.brz_order_shipments")
    .awaitTermination()
)

In [0]:
# display(
#     spark.sql(
#         f"SELECT COUNT(*) FROM {catalog_name}.bronze.brz_order_shipments"
#     )
# )

COUNT(*)
679731


In [0]:
# df = spark.table(f"{catalog_name}.bronze.brz_order_shipments")

# df.printSchema()
# display(df.limit(20))

root
 |-- order_dt: date (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



order_dt,shipment_id,order_id,carrier,_rescued_data,ingest_timestamp,source_file
2025-08-01,S643611,643611,BlueDart,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643612,643612,AusPost,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643613,643613,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643614,643614,XpressBees,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643615,643615,Delhivery,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643616,643616,BlueDart,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643617,643617,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643618,643618,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643619,643619,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643620,643620,DHL,null,2026-06-15T07:30:13.815Z,abfss://ecomm-raw-data@stgecommjackyind.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
